In [1]:
import os
import glob
import pandas as pd
import numpy as np
%run ./GetData.ipynb

In [2]:
base_dir = "/home/msp25gd/Downloads"
input_fits_path = base_dir
output_meta_dir = os.path.join(base_dir, "res", "meta")
os.makedirs(output_meta_dir, exist_ok=True)

# ESO downloads in your folder are named ADP.<timestamp> (no .fits extension).
file_list = sorted(
    f for f in glob.glob(os.path.join(input_fits_path, "ADP.*")) if os.path.isfile(f)
 )
print(f"Matched {len(file_list)} input files")
if len(file_list) == 0:
    raise FileNotFoundError(
        f"No input files found in {input_fits_path}. Expected files like ADP.*"
    )

np.save(os.path.join(output_meta_dir, "fits_list.npy"), file_list)

allkeys = ['OBJECT', 'RA', 'DEC', 'EXPTIME', 'MJD-OBS', 'MJD-END',
        'WAVELMIN', 'WAVELMAX', 'SPEC_BIN', 'SNR', 'SPEC_RES']

df = pd.DataFrame(data={'OBJECT': [], 'RA': [], 'DEC': [], 'EXPTIME': [], 'MJD-OBS': [], 'MJD-END': [],
        'WAVELMIN': [], 'WAVELMAX': [], 'SPEC_BIN': [], 'SNR': [], 'SPEC_RES': [], 'Checked': []})

# In case error occurs during download
loaded = False
# loaded = True

if loaded is True:
# ========== Import main_df ===================================================
    df = pd.read_pickle(os.path.join(output_meta_dir, 'metadata.pkl'))
    print(df)

    start = df[df['Checked'] == True].iloc[[-1]].index[0]
# =============================================================================

MetaData = GetSpectra()

for i, file in enumerate(file_list):

    if loaded is True:
        if i <= start:
            continue

    df = MetaData.get_metadata(i, file, allkeys, main_df=df)

    print('{}/{}'.format(i + 1, len(file_list)))

    if i % 20 == 0:
        df.to_pickle(os.path.join(output_meta_dir, 'metadata.pkl'))
        MetaData.corrupt.to_pickle(os.path.join(output_meta_dir, 'corrupt.pkl'))
        print('Saved')
        print('----------------------')

df.to_pickle(os.path.join(output_meta_dir, 'metadata.pkl'))
MetaData.corrupt.to_pickle(os.path.join(output_meta_dir, 'corrupt.pkl'))
print('Saved')
print('----------------------')
print(df)
print(MetaData.corrupt)

Matched 84188 input files
1/84188
Saved
----------------------
2/84188
3/84188
4/84188
5/84188
6/84188
7/84188
8/84188
9/84188
10/84188
11/84188
12/84188
13/84188
14/84188
15/84188
16/84188
17/84188
18/84188
19/84188
20/84188
21/84188
Saved
----------------------
22/84188
23/84188
24/84188
25/84188
26/84188
27/84188
28/84188
29/84188
30/84188
31/84188
32/84188
33/84188
34/84188
35/84188
36/84188
37/84188
38/84188
39/84188
40/84188
41/84188
Saved
----------------------
42/84188
43/84188
44/84188
45/84188
46/84188
47/84188
48/84188
49/84188
50/84188
51/84188
52/84188
53/84188
54/84188
55/84188
56/84188
57/84188
58/84188
59/84188
60/84188
61/84188
Saved
----------------------
62/84188
63/84188
64/84188
65/84188
66/84188
67/84188
68/84188
69/84188
70/84188
71/84188
72/84188
73/84188
74/84188
75/84188
76/84188
77/84188
78/84188
79/84188
80/84188
81/84188
Saved
----------------------
82/84188
83/84188
84/84188
85/84188
86/84188
87/84188
88/84188
89/84188
90/84188
91/84188
92/84188
93/84188
9

In [3]:
df

,OBJECT,RA,DEC,EXPTIME,MJD-OBS,MJD-END,WAVELMIN,WAVELMAX,SPEC_BIN,SNR,SPEC_RES,Checked
0,HE 0048-6408,12.688136,-63.86430,999.9990,58728.269293,58728.280867,565.503800,946.381600,0.001800,60.789400,56990.0,True
1,HE 0048-6408,12.688136,-63.86430,1000.0017,58728.269339,58728.280913,373.211600,499.979100,0.001500,39.243300,53750.0,True
2,QSO B2139-4433,325.592422,-44.32452,5393.0002,58730.089919,58730.152338,458.307200,668.649800,0.002800,2.860200,42310.0,True
3,QSO B2139-4433,325.592422,-44.32452,5395.9983,58730.089966,58730.152420,328.194600,456.297700,0.002700,0.301700,40970.0,True
4,QSO B2139-4433,325.592345,-44.32473,5393.0010,58730.154759,58730.217178,458.307200,668.649800,0.002800,2.355700,42310.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...
84183,HD 39746,88.918130,27.71553,254.9974,60949.350545,60949.354604,373.216133,499.980854,0.001474,127.932585,71050.0,True
84184,HD 41690,91.912330,21.87303,459.9996,60951.366177,60951.371501,472.644600,683.499000,0.001400,360.236600,107200.0,True
84185,HD 41690,91.912330,21.87303,459.9996,60951.366213,60951.371537,304.428900,391.673500,0.001200,184.324700,71050.0,True
84186,HD 39746,88.917971,27.71550,299.9995,60951.357992,60951.361464,472.644600,683.499000,0.001400,388.034000,107200.0,True


In [ ]:
# Calcium II doublet central wavelengths
ca_k_angstrom = 3933.66
ca_h_angstrom = 3968.47

# Detect whether WAVELMIN/WAVELMAX are likely in Angstrom or nm
if df[["WAVELMIN", "WAVELMAX"]].median().max() < 1000:
    ca_k = ca_k_angstrom / 10.0  # nm
    ca_h = ca_h_angstrom / 10.0  # nm
else:
    ca_k = ca_k_angstrom          # Angstrom
    ca_h = ca_h_angstrom          # Angstrom

# Keep only observations whose wavelength range includes both Ca II lines
line_min = min(ca_k, ca_h)
line_max = max(ca_k, ca_h)
mask_ca_doublet = (df["WAVELMIN"] <= line_min) & (df["WAVELMAX"] >= line_max)

df_ca_doublet = df.loc[mask_ca_doublet].copy()

print(f"Kept {len(df_ca_doublet)} of {len(df)} observations covering the Ca II doublet")
df_ca_doublet.head()